In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (per rule)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# 1) For each file report row counts and unique cmdb_id values (up to 50)
def summarize_file(df, name, max_ids=50):
    rows = df.shape[0]
    unique_ids = df['cmdb_id'].dropna().unique()
    unique_count = len(unique_ids)
    # truncated id list string
    id_list_str = ", ".join(map(str, unique_ids[:max_ids]))
    if unique_count > max_ids:
        id_list_str += ", ..."
    return pd.DataFrame([{
        'file': name,
        'row_count': rows,
        'unique_cmdb_count': unique_count,
        'unique_cmdb_ids_sample': id_list_str
    }])

summary_metric = summarize_file(metric, 'metric.csv')
summary_trace = summarize_file(trace, 'trace.csv')
summary_log = summarize_file(log, 'log.csv')
summary_error_logs = summarize_file(error_logs, 'error_logs.csv')

file_summary = pd.concat([summary_metric, summary_trace, summary_log, summary_error_logs], ignore_index=True)

# 2) metric.csv: unique kpi_name values (up to 50) and compute global P95 per (cmdb_id, kpi_name)
metric_kpi_unique = pd.DataFrame({'kpi_name': pd.Series(sorted(metric['kpi_name'].dropna().unique()))})
metric_kpi_unique = metric_kpi_unique.head(50)

# Compute global p95 and count per (cmdb_id, kpi_name) using full series
metric_p95 = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count_of_points='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)

# Keep requested columns and order by p95_value desc, limit top 20
metric_p95['p95_value'] = metric_p95['p95_value'].astype(float).round(6)
metric_p95_top20 = metric_p95.sort_values('p95_value', ascending=False).loc[:, ['cmdb_id', 'kpi_name', 'p95_value', 'count_of_points']].head(20)

# 3) trace.csv: unique trace_name values (up to 50) and counts per (cmdb_id, trace_name) top 20 by count
trace_unique = pd.DataFrame({'trace_name': pd.Series(sorted(trace['trace_name'].dropna().unique()))}).head(50)

trace_counts = (
    trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
trace_counts_top20 = trace_counts.head(20)

# 4) log.csv: unique log_name values (up to 50) and counts per (cmdb_id, log_name) top 20 by count
log_unique = pd.DataFrame({'log_name': pd.Series(sorted(log['log_name'].dropna().unique()))}).head(50)

log_counts = (
    log
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
log_counts_top20 = log_counts.head(20)

# 5) Check which candidate root-cause components appear in cmdb_id columns of metric, trace, log
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']
files_cmdb_sets = {
    'metric': set(metric['cmdb_id'].dropna().unique()),
    'trace': set(trace['cmdb_id'].dropna().unique()),
    'log': set(log['cmdb_id'].dropna().unique())
}

presence_rows = []
for comp in candidates:
    presence_rows.append({
        'component': comp,
        'in_metric': comp in files_cmdb_sets['metric'],
        'in_trace': comp in files_cmdb_sets['trace'],
        'in_log': comp in files_cmdb_sets['log'],
        'present_any': (comp in files_cmdb_sets['metric']) or (comp in files_cmdb_sets['trace']) or (comp in files_cmdb_sets['log'])
    })
candidate_presence = pd.DataFrame(presence_rows)

# Compact textual summaries:
# - file_summary (one row per file)
# - metric_kpi_unique (up to 50)
# - metric_p95_top20
# - trace_unique (up to 50) and trace_counts_top20
# - log_unique (up to 50) and log_counts_top20
# - candidate_presence

file_summary, metric_kpi_unique, metric_p95_top20, trace_unique, trace_counts_top20, log_unique, log_counts_top20, candidate_presence

```
Out[1]:
```
```python
# Compose a compact plain-English summary using previously loaded results.
# Reuse variables from prior execution: file_summary, metric_kpi_unique, metric_p95_top20,
# trace_unique, trace_counts_top20, log_unique, log_counts_top20, candidate_presence

# File-level summaries
metric_row = file_summary.loc[file_summary['file']=='metric.csv'].iloc[0]
trace_row = file_summary.loc[file_summary['file']=='trace.csv'].iloc[0]
log_row = file_summary.loc[file_summary['file']=='log.csv'].iloc[0]
err_row = file_summary.loc[file_summary['file']=='error_logs.csv'].iloc[0]

# Metric KPI list (compact)
kpi_list = ", ".join(metric_kpi_unique['kpi_name'].tolist())

# Top 5 metric series by P95 (compact lines)
top_metric_rows = metric_p95_top20.head(5)
top_metric_lines = []
for i, r in top_metric_rows.reset_index(drop=True).iterrows():
    top_metric_lines.append(f"{i+1}) {r['cmdb_id']} | {r['kpi_name']} | p95={r['p95_value']:.6g} | points={int(r['count_of_points'])}")

# Trace summary: count of unique trace names and top 3 trace counts
trace_unique_count = trace_unique.shape[0]
trace_top = trace_counts_top20.head(3)
trace_top_lines = []
for i, r in trace_top.reset_index(drop=True).iterrows():
    trace_top_lines.append(f"{i+1}) {r['cmdb_id']} | {r['trace_name']} | count={int(r['count'])}")

# Log summary: unique log names and top 3 log counts
log_unique_list = ", ".join(log_unique['log_name'].tolist())
log_top = log_counts_top20.head(3)
log_top_lines = []
for i, r in log_top.reset_index(drop=True).iterrows():
    log_top_lines.append(f"{i+1}) {r['cmdb_id']} | {r['log_name']} | count={int(r['count'])}")

# Candidate components presence
present = candidate_presence[candidate_presence['present_any']].loc[:, 'component'].tolist()
absent = candidate_presence[~candidate_presence['present_any']].loc[:, 'component'].tolist()

summary = (
    "Telemetry summary (concise):\n\n"
    "1) File counts and cmdb_id samples:\n"
    f"- metric.csv: {int(metric_row['row_count'])} rows, {int(metric_row['unique_cmdb_count'])} unique cmdb_ids (sample: {metric_row['unique_cmdb_ids_sample']})\n"
    f"- trace.csv:  {int(trace_row['row_count'])} rows, {int(trace_row['unique_cmdb_count'])} unique cmdb_ids (sample: {trace_row['unique_cmdb_ids_sample']})\n"
    f"- log.csv:    {int(log_row['row_count'])} rows, {int(log_row['unique_cmdb_count'])} unique cmdb_ids (sample: {log_row['unique_cmdb_ids_sample']})\n"
    f"- error_logs.csv: {int(err_row['row_count'])} rows, {int(err_row['unique_cmdb_count'])} unique cmdb_ids\n\n"
    "2) metric.csv KPIs and top series by global P95:\n"
    f"- Unique kpi_name values (sample/up to 50): {kpi_list}\n"
    "- Top metric series by global P95 (top 5):\n"
    + "\n".join("   " + line for line in top_metric_lines) + "\n\n"
    "3) trace.csv summary:\n"
    f"- Unique trace_name count: {trace_unique_count} (sample names available)\n"
    "- Top trace counts (top 3):\n"
    + "\n".join("   " + line for line in trace_top_lines) + "\n\n"
    "4) log.csv summary:\n"
    f"- Unique log_name values: {log_unique_list}\n"
    "- Top log counts (top 3):\n"
    + "\n".join("   " + line for line in log_top_lines) + "\n\n"
    "5) Candidate root-cause components presence:\n"
    f"- Present in any of metric/trace/log: {', '.join(present) if present else 'None'}\n"
    f"- Absent from all three files: {', '.join(absent) if absent else 'None'}\n\n"
    "Notes: error_logs.csv is empty. Metric global P95 was computed per (cmdb_id, kpi_name) using the full available series.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count  unique_cmdb_count                             unique_cmdb_ids_sample
0      metric.csv       1750                 11  adservice, cartservice, checkoutservice, curre...
1       trace.csv       3800                  8  checkoutservice, currencyservice, emailservice...
2         log.csv        460                 10  adservice, cartservice, checkoutservice, curre...
3  error_logs.csv          0                  0                                                   ,      kpi_name
0         cpu
1      diskio
2       error
3  latency-50
4  latency-90
5         mem
6      socket
7    workload,                   cmdb_id  kpi_name     p95_value  count_of_points
15        checkoutservice    diskio  1.056899e+10               25
18        checkoutservice       mem  2.683793e+08               25
4               adservice       mem  1.063802e+08               25
24        currencyservice       mem  4.580035e+07               25
57  recommendationservice       mem  4.480175e+07               25
11            cartservice       mem  4.377885e+07               25
31           emailservice       mem  4.357595e+07               25
44         paymentservice       mem  3.653932e+07               25
38               frontend       mem  1.554960e+07               25
50  productcatalogservice       mem  1.050280e+07               25
67        shippingservice       mem  8.700027e+06               25
62                  redis       mem  3.052448e+06               25
1               adservice    diskio  2.400183e+05               25
61                  redis    diskio  1.218944e+04               25
52  productcatalogservice  workload  1.521813e+02               25
26        currencyservice  workload  8.350470e+01               25
21        currencyservice       cpu  3.317800e+01               25
40               frontend  workload  2.768791e+01               25
13            cartservice  workload  2.670336e+01               25
39               frontend    socket  2.400000e+01               25,                                       trace_name
0       trace.from_checkoutservice.duration_mean
1        trace.from_checkoutservice.duration_p95
2          trace.from_checkoutservice.error_rate
3           trace.from_checkoutservice.row_count
4       trace.from_frontendservice.duration_mean
..                                           ...
39      trace.to_productcatalogservice.row_count
40  trace.to_recommendationservice.duration_mean
41   trace.to_recommendationservice.duration_p95
42     trace.to_recommendationservice.error_rate
43      trace.to_recommendationservice.row_count

[44 rows x 1 columns],             cmdb_id                                trace_name  count
0   checkoutservice  trace.from_checkoutservice.duration_mean     25
1   checkoutservice   trace.from_checkoutservice.duration_p95     25
2   checkoutservice     trace.from_checkoutservice.error_rate     25
3   checkoutservice      trace.from_checkoutservice.row_count     25
4   checkoutservice  trace.from_frontendservice.duration_mean     25
5   checkoutservice   trace.from_frontendservice.duration_p95     25
6   checkoutservice     trace.from_frontendservice.error_rate     25
7   checkoutservice      trace.from_frontendservice.row_count     25
8   checkoutservice             trace.from_root.duration_mean     25
9   checkoutservice              trace.from_root.duration_p95     25
10  checkoutservice                trace.from_root.error_rate     25
11  checkoutservice                 trace.from_root.row_count     25
12  checkoutservice    trace.to_checkoutservice.duration_mean     25
13  checkoutservice     trace.to_checkoutservice.duration_p95     25
14  checkoutservice       trace.to_checkoutservice.error_rate     25
15  checkoutservice        trace.to_checkoutservice.row_count     25
16  checkoutservice    trace.to_currencyservice.duration_mean     25
17  checkoutservice     trace.to_currencyservice.duration_p95     25
18  checkoutservice       trace.to_currencyservice.error_rate     25
19  checkoutservice        trace.to_currencyservice.row_count     25,           log_name
0  log.error_count
1    log.row_count,                   cmdb_id         log_name  count
0               adservice  log.error_count     25
1               adservice    log.row_count     25
2             cartservice  log.error_count     25
3             cartservice    log.row_count     25
4         checkoutservice  log.error_count     25
5         checkoutservice    log.row_count     25
6         currencyservice  log.error_count     25
7         currencyservice    log.row_count     25
8            emailservice  log.error_count     25
9            emailservice    log.row_count     25
10               frontend  log.error_count     25
11               frontend    log.row_count     25
12         paymentservice  log.error_count     25
13         paymentservice    log.row_count     25
14  recommendationservice  log.error_count     25
15  recommendationservice    log.row_count     25
19        shippingservice    log.row_count     25
18        shippingservice  log.error_count     25
17                  redis    log.row_count      5
16                  redis  log.error_count      5,                component  in_metric  in_trace  in_log  present_any
0        checkoutservice       True      True    True         True
1        currencyservice       True      True    True         True
2           emailservice       True      True    True         True
3  productcatalogservice       True      True   False         True
4  recommendationservice       True      True    True         True)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' DataFrame loaded earlier in the session.
# Ensure timestamp is parsed as UTC datetime (rule 10)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 and P5 for every (cmdb_id, kpi_name) using full metric series (do NOT filter by time window)
pcts = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05),
    total_points='count'
).reset_index()

# Restrict to candidate components only for downstream steps
pcts_candidates = pcts[pcts['cmdb_id'].isin(candidates)].copy()

# 2) Filter metric.csv to the incident window (UTC)
start = pd.to_datetime("2024-01-20 16:08:45", utc=True)
end   = pd.to_datetime("2024-01-20 16:38:45", utc=True)
metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

# 3) For each (cmdb_id, kpi_name) of the candidate components compute points_in_window, anomaly_count, earliest anomaly timestamp
# Merge window rows with global thresholds (p95/p5) so each row knows its series thresholds
window_candidates = metric_window[metric_window['cmdb_id'].isin(candidates)].merge(
    pcts_candidates[['cmdb_id', 'kpi_name', 'p95', 'p5']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Define anomaly condition per rule:
# if kpi_name == 'workload' then anomaly if value <= P5, else anomaly if value >= P95
def is_anomaly_row(df):
    cond_workload = (df['kpi_name'] == 'workload') & (df['value'] <= df['p5'])
    cond_other = (df['kpi_name'] != 'workload') & (df['value'] >= df['p95'])
    return cond_workload | cond_other

if not window_candidates.empty:
    window_candidates['is_anomaly'] = is_anomaly_row(window_candidates).fillna(False)
else:
    window_candidates['is_anomaly'] = pd.Series(dtype=bool)

# Aggregate per series in-window
agg_window = window_candidates.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    points_in_window=('value', 'count'),
    anomaly_count=('is_anomaly', 'sum'),
    earliest_anomaly_timestamp=('timestamp', lambda x: x.min() if x.size>0 else pd.NaT)
).reset_index()

# Ensure series with zero points in window (but present in pcts_candidates) are included with zeros
agg_full = pcts_candidates.merge(
    agg_window,
    on=['cmdb_id', 'kpi_name'],
    how='left',
    suffixes=('', '_win')
)

# Fill NaNs for series without window points
agg_full['points_in_window'] = agg_full['points_in_window'].fillna(0).astype(int)
agg_full['anomaly_count'] = agg_full['anomaly_count'].fillna(0).astype(int)
# earliest_anomaly_timestamp may be NaT - keep as is
# Compute anomaly_rate safely
agg_full['anomaly_rate'] = np.where(
    agg_full['points_in_window'] > 0,
    agg_full['anomaly_count'] / agg_full['points_in_window'],
    0.0
)

# Format earliest_anomaly_timestamp as UTC ISO string, or empty if none
def fmt_ts(x):
    if pd.isna(x):
        return pd.NaT
    # ensure tz-aware UTC and format
    return pd.to_datetime(x).tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ')

agg_full['earliest_anomaly_timestamp_utc'] = agg_full['earliest_anomaly_timestamp'].apply(fmt_ts)

# Select and order columns as requested
result_table = agg_full.loc[:, [
    'cmdb_id', 'kpi_name', 'p95', 'p5', 'points_in_window', 'anomaly_count', 'anomaly_rate', 'earliest_anomaly_timestamp_utc'
]].copy()

# Sort by anomaly_rate desc then anomaly_count desc, limit to top 20
result_table = result_table.sort_values(['anomaly_rate', 'anomaly_count'], ascending=[False, False]).reset_index(drop=True)
result_table_top20 = result_table.head(20)

# 4) Short list of which candidate components have any anomalies and their total anomaly counts across KPIs
anomalies_by_component = result_table.groupby('cmdb_id', dropna=False)['anomaly_count'].sum().reset_index()
anomalous_components = anomalies_by_component[anomalies_by_component['anomaly_count'] > 0].sort_values('anomaly_count', ascending=False)

# Keep outputs compact
result_table_top20, anomalous_components, result_table.shape

```
Out[2]:
```
# Compose a concise plain-English summary based on prior computation results.
summary = (
    "Summary of anomaly analysis for candidate components (window: 2024-01-20 16:08:45 to 2024-01-20 16:38:45 UTC):\n\n"
    "1) Method: global P95 and P5 were computed per (cmdb_id, kpi_name) using the full metric.csv. "
    "Within the incident window, a point is anomalous if: workload <= P5, otherwise value >= P95.\n\n"
    "2) Which components showed anomalies (total anomaly counts across KPIs):\n"
    "- emailservice: 60 anomalies\n"
    "- recommendationservice: 60 anomalies\n"
    "- currencyservice: 35 anomalies\n"
    "- productcatalogservice: 35 anomalies\n"
    "- checkoutservice: 23 anomalies\n\n"
    "3) Highest-severity series (100% of in-window points anomalous, earliest anomaly at 2024-01-20T16:11:00Z):\n"
    "- currencyservice | socket (25/25)\n"
    "- emailservice    | diskio  (25/25)\n"
    "- emailservice    | socket  (25/25)\n"
    "- productcatalogservice | socket (25/25)\n"
    "- recommendationservice | diskio  (25/25)\n"
    "- recommendationservice | socket  (25/25)\n\n"
    "4) Other notable series:\n"
    "- checkoutservice | socket: 11 anomalies out of 25 points (44%), earliest anomaly 2024-01-20T16:11:00Z\n"
    "- Several other checkoutservice, currencyservice, and emailservice KPIs had small counts of anomalies (typically 2/25 = 8%), earliest anomaly times also at 2024-01-20T16:11:00Z.\n\n"
    "5) Overall conclusion: All five candidate components (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice) show anomalies in the incident window. "
    "The most severe and immediate signals are socket and diskio KPIs for currencyservice, emailservice, productcatalogservice, and recommendationservice (complete in-window anomaly rates). "
    "Checkoutservice shows elevated socket anomalies but lower rates on other KPIs.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name           p95            p5  points_in_window  anomaly_count  anomaly_rate earliest_anomaly_timestamp_utc
0         currencyservice      socket  4.000000e+00  4.000000e+00                25             25          1.00           2024-01-20T16:11:00Z
1            emailservice      diskio  0.000000e+00  0.000000e+00                25             25          1.00           2024-01-20T16:11:00Z
2            emailservice      socket  3.000000e+00  3.000000e+00                25             25          1.00           2024-01-20T16:11:00Z
3   productcatalogservice      socket  4.000000e+00  4.000000e+00                25             25          1.00           2024-01-20T16:11:00Z
4   recommendationservice      diskio  0.000000e+00  0.000000e+00                25             25          1.00           2024-01-20T16:11:00Z
5   recommendationservice      socket  4.000000e+00  4.000000e+00                25             25          1.00           2024-01-20T16:11:00Z
6         checkoutservice      socket  1.200000e+01  9.000000e+00                25             11          0.44           2024-01-20T16:11:00Z
7         checkoutservice         cpu  1.920772e+01  2.787340e-01                25              2          0.08           2024-01-20T16:11:00Z
8         checkoutservice      diskio  1.056899e+10  0.000000e+00                25              2          0.08           2024-01-20T16:11:00Z
9         checkoutservice  latency-50  2.494769e-01  6.965461e-02                25              2          0.08           2024-01-20T16:11:00Z
10        checkoutservice  latency-90  5.492023e-01  2.001502e-01                25              2          0.08           2024-01-20T16:11:00Z
11        checkoutservice         mem  2.683793e+08  1.115552e+07                25              2          0.08           2024-01-20T16:11:00Z
12        checkoutservice    workload  1.119477e+00  6.073000e-01                25              2          0.08           2024-01-20T16:11:00Z
13        currencyservice         cpu  3.317800e+01  2.957666e+01                25              2          0.08           2024-01-20T16:11:00Z
14        currencyservice  latency-50  3.958248e-03  3.715922e-03                25              2          0.08           2024-01-20T16:11:00Z
15        currencyservice  latency-90  4.896989e-02  3.431252e-02                25              2          0.08           2024-01-20T16:11:00Z
16        currencyservice         mem  4.580035e+07  4.467268e+07                25              2          0.08           2024-01-20T16:11:00Z
17        currencyservice    workload  8.350470e+01  7.592818e+01                25              2          0.08           2024-01-20T16:11:00Z
18           emailservice         cpu  2.957889e-01  2.471725e-01                25              2          0.08           2024-01-20T16:11:00Z
19           emailservice  latency-50  3.202963e-03  3.000000e-03                25              2          0.08           2024-01-20T16:11:00Z,                  cmdb_id  anomaly_count
2           emailservice             60
4  recommendationservice             60
1        currencyservice             35
3  productcatalogservice             35
0        checkoutservice             23, (33, 8))```
```

In [ ]:
# In[3]:

import re
import pandas as pd
import numpy as np

# Ensure trace DataFrame is loaded and timestamp is utc datetime
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Global P95 per (cmdb_id, trace_name) using full trace.csv
trace_p95 = (
    trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter trace to incident window (UTC)
start = pd.to_datetime("2024-01-20 16:08:45", utc=True)
end = pd.to_datetime("2024-01-20 16:38:45", utc=True)
trace_win = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# 4) Extract direction and peer_service from trace_name
def parse_direction_peer(name):
    m_to = re.search(r'\bto_([^.\s]+)', name)
    m_from = re.search(r'\bfrom_([^.\s]+)', name)
    if m_to:
        return 'to', m_to.group(1)
    if m_from:
        return 'from', m_from.group(1)
    return 'other', ''

parsed = trace_win['trace_name'].apply(parse_direction_peer)
trace_win[['direction', 'peer_service']] = pd.DataFrame(parsed.tolist(), index=trace_win.index)

# 3) Merge global p95 into window rows and mark anomalies (value >= p95)
trace_win = trace_win.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')
# Treat missing p95 as a non-anomalous threshold (no anomaly)
trace_win['is_anomaly'] = (trace_win['value'] >= trace_win['p95']).fillna(False)

# Aggregate per (cmdb_id, trace_name, direction, peer_service)
agg_basic = (
    trace_win
    .groupby(['cmdb_id', 'trace_name', 'direction', 'peer_service'], dropna=False)
    .agg(
        p95=('p95', 'first'),
        points_in_window=('value', 'count'),
        anomaly_count=('is_anomaly', 'sum')
    )
    .reset_index()
)

# Compute earliest anomaly timestamp per (cmdb_id, trace_name)
anom_ts = (
    trace_win[trace_win['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'])['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp')
)

# Merge earliest anomaly timestamp into agg_basic
agg = agg_basic.merge(anom_ts, on=['cmdb_id', 'trace_name'], how='left')

# Compute anomaly_rate and format earliest_anomaly_timestamp_utc
agg['anomaly_rate'] = np.where(agg['points_in_window']>0, agg['anomaly_count'] / agg['points_in_window'], 0.0)
agg['earliest_anomaly_timestamp_utc'] = agg['earliest_anomaly_timestamp'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
agg['earliest_anomaly_timestamp_utc'] = agg['earliest_anomaly_timestamp_utc'].where(agg['earliest_anomaly_timestamp'].notna(), pd.NA)

# Select and order requested columns
result = agg.loc[:, ['cmdb_id', 'trace_name', 'direction', 'peer_service', 'p95', 'points_in_window', 'anomaly_count', 'anomaly_rate', 'earliest_anomaly_timestamp_utc']].copy()

# Sort by anomaly_count desc then anomaly_rate desc, limit top 20
result_top20 = result.sort_values(['anomaly_count', 'anomaly_rate'], ascending=[False, False]).head(20).reset_index(drop=True)

# Peer services that appear as anomalous targets (anomaly_count>0 & peer_service not blank)
peer_anoms = (
    result[(result['peer_service'].notna()) & (result['peer_service'] != '') & (result['anomaly_count']>0)]
    .groupby('peer_service', dropna=False)['anomaly_count']
    .sum()
    .reset_index()
    .sort_values('anomaly_count', ascending=False)
)

peer_anoms_head = peer_anoms.head(20)

# Return compact results
result_top20, peer_anoms_head

```
Out[3]:
```
```python
# Plain-English summary of the trace anomaly analysis (compact)
summary = (
    "Trace anomaly summary (window: 2024-01-20 16:08:45 to 2024-01-20 16:38:45 UTC):\n\n"
    "Method: computed global P95 per (cmdb_id, trace_name) from the full trace.csv. "
    "Within the incident window, a trace point is marked anomalous when value >= that global P95.\n\n"
    "Key findings:\n"
    "- Many trace.error_rate series are fully anomalous in-window (25/25 points, anomaly_rate=1.0). "
    "Examples (earliest anomaly timestamp observed: 2024-01-20T16:11:00Z):\n"
    "  * checkoutservice: trace.from_checkoutservice.error_rate, trace.from_frontendservice.error_rate, trace.from_root.error_rate, trace.to_checkoutservice.error_rate, trace.to_currencyservice.error_rate, trace.to_emailservice.error_rate, trace.to_paymentservice.error_rate, trace.to_productcatalogservice.error_rate\n"
    "  * currencyservice, emailservice, frontendservice and others show the same pattern for many trace.error_rate names (each with 25/25 anomalous points and earliest anomaly at 2024-01-20T16:11:00Z).\n\n"
    "Peer services flagged by anomalies (peer_service -> total anomaly_count across traces):\n"
    "- root: 322\n"
    "- checkoutservice: 254\n"
    "- frontendservice: 221\n"
    "- recommendationservice: 176\n"
    "- productcatalogservice: 147\n"
    "- currencyservice: 114\n"
    "- emailservice: 85\n"
    "- paymentservice: 85\n\n"
    "Summary conclusion: there is a broad, high-severity elevation in trace error rates across many services and directions during the incident window, with the highest anomalous counts associated with the 'root' peer and heavy counts seen for checkoutservice and frontendservice. These peer services (and the high-count trace.error_rate series) are the most prominent anomalous signals to investigate further."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                 trace_name direction           peer_service  p95  points_in_window  anomaly_count  anomaly_rate earliest_anomaly_timestamp_utc
0   checkoutservice      trace.from_checkoutservice.error_rate      from        checkoutservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
1   checkoutservice      trace.from_frontendservice.error_rate      from        frontendservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
2   checkoutservice                 trace.from_root.error_rate      from                   root  0.0                25             25           1.0           2024-01-20T16:11:00Z
3   checkoutservice        trace.to_checkoutservice.error_rate        to        checkoutservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
4   checkoutservice        trace.to_currencyservice.error_rate        to        currencyservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
5   checkoutservice           trace.to_emailservice.error_rate        to           emailservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
6   checkoutservice         trace.to_paymentservice.error_rate        to         paymentservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
7   checkoutservice  trace.to_productcatalogservice.error_rate        to  productcatalogservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
8   currencyservice      trace.from_checkoutservice.error_rate      from        checkoutservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
9   currencyservice      trace.from_frontendservice.error_rate      from        frontendservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
10  currencyservice                 trace.from_root.error_rate      from                   root  0.0                25             25           1.0           2024-01-20T16:11:00Z
11     emailservice      trace.from_checkoutservice.error_rate      from        checkoutservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
12     emailservice                 trace.from_root.error_rate      from                   root  0.0                25             25           1.0           2024-01-20T16:11:00Z
13  frontendservice      trace.from_frontendservice.error_rate      from        frontendservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
14  frontendservice                 trace.from_root.error_rate      from                   root  0.0                25             25           1.0           2024-01-20T16:11:00Z
15  frontendservice        trace.to_checkoutservice.error_rate        to        checkoutservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
16  frontendservice        trace.to_currencyservice.error_rate        to        currencyservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
17  frontendservice        trace.to_frontendservice.error_rate        to        frontendservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
18  frontendservice  trace.to_productcatalogservice.error_rate        to  productcatalogservice  0.0                25             25           1.0           2024-01-20T16:11:00Z
19  frontendservice  trace.to_recommendationservice.error_rate        to  recommendationservice  0.0                25             25           1.0           2024-01-20T16:11:00Z,             peer_service  anomaly_count
7                   root            322
0        checkoutservice            254
3        frontendservice            221
6  recommendationservice            176
5  productcatalogservice            147
1        currencyservice            114
2           emailservice             85
4         paymentservice             85)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse previously loaded dataframes and computed vars:
# metric, trace, trace_p95, trace_win, result_table (metric per-series for candidates)
# Ensure timestamps parsed (they should be from prior steps)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Recompute global trace P95 if trace_p95 not present / to be safe (rule 9 allows reuse but recomputing is idempotent)
trace_p95 = trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.95).reset_index(name='p95')

# Incident window
start = pd.to_datetime("2024-01-20 16:08:45", utc=True)
end   = pd.to_datetime("2024-01-20 16:38:45", utc=True)

# 1) Filter trace to window and mark anomalies value >= global P95
trace_win = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()
trace_win = trace_win.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')
trace_win['is_anomaly'] = (trace_win['value'] >= trace_win['p95']).fillna(False)

# Parse direction and peer_service if not already present
import re
def parse_direction_peer(name):
    m_to = re.search(r'\bto_([^.\s]+)', name)
    m_from = re.search(r'\bfrom_([^.\s]+)', name)
    if m_to:
        return 'to', m_to.group(1)
    if m_from:
        return 'from', m_from.group(1)
    return 'other', ''
parsed = trace_win['trace_name'].apply(parse_direction_peer)
trace_win[['direction', 'peer_service']] = pd.DataFrame(parsed.tolist(), index=trace_win.index)

# Candidate components
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 2) Metric anomaly counts: reuse result_table (which was built earlier for candidates)
# If result_table is missing, compute metric anomalies here similarly; otherwise use it.
try:
    _ = result_table  # use existing
except NameError:
    # compute metric series p95/p5 for all, then window anomalies for candidates (fallback, should not run normally)
    pcts = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].agg(
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()
    metric_window = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()
    window_candidates = metric_window[metric_window['cmdb_id'].isin(candidates)].merge(
        pcts, on=['cmdb_id','kpi_name'], how='left'
    )
    window_candidates['is_anomaly'] = ((window_candidates['kpi_name']=='workload') & (window_candidates['value'] <= window_candidates['p5'])) | \
                                      ((window_candidates['kpi_name']!='workload') & (window_candidates['value'] >= window_candidates['p95']))
    result_table = (window_candidates.groupby(['cmdb_id','kpi_name'], dropna=False)
                    .agg(p95=('p95','first'), p5=('p5','first'),
                         points_in_window=('value','count'),
                         anomaly_count=('is_anomaly','sum'),
                         earliest_anomaly_timestamp=('timestamp', lambda x: x.min() if x.size>0 else pd.NaT))
                    .reset_index())
    # format earliest timestamp
    result_table['earliest_anomaly_timestamp_utc'] = pd.to_datetime(result_table['earliest_anomaly_timestamp']).dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    result_table['earliest_anomaly_timestamp_utc'] = result_table['earliest_anomaly_timestamp_utc'].where(result_table['earliest_anomaly_timestamp'].notna(), pd.NA)

# metric_anomaly_count per component (sum anomaly_count across KPIs inside window)
metric_by_comp = result_table.groupby('cmdb_id', dropna=False)['anomaly_count'].sum().reset_index().rename(columns={'anomaly_count':'metric_anomaly_count'})

# earliest metric anomaly timestamp per component
# use earliest non-null earliest_anomaly_timestamp_utc among series with anomaly_count>0
if 'earliest_anomaly_timestamp_utc' in result_table.columns:
    # convert to datetime for min
    tmp = result_table[result_table['anomaly_count']>0].copy()
    if not tmp.empty:
        tmp['earliest_ts_dt'] = pd.to_datetime(tmp['earliest_anomaly_timestamp_utc'], utc=True, errors='coerce')
        earliest_metric_ts = tmp.groupby('cmdb_id', dropna=False)['earliest_ts_dt'].min().reset_index()
        earliest_metric_ts['earliest_metric_anomaly_timestamp_utc'] = earliest_metric_ts['earliest_ts_dt'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        earliest_metric_ts = earliest_metric_ts[['cmdb_id','earliest_metric_anomaly_timestamp_utc']]
    else:
        earliest_metric_ts = pd.DataFrame(columns=['cmdb_id','earliest_metric_anomaly_timestamp_utc'])
else:
    earliest_metric_ts = pd.DataFrame(columns=['cmdb_id','earliest_metric_anomaly_timestamp_utc'])

# 2) Trace aggregates per candidate
rows = []
for comp in candidates:
    # trace_to_anomaly_count: traces where direction=='to' and peer_service == comp and is_anomaly
    to_mask = (trace_win['direction']=='to') & (trace_win['peer_service']==comp) & (trace_win['is_anomaly'])
    trace_to_anom = int(to_mask.sum())
    # trace_from_anomaly_count: where cmdb_id == comp and trace_name starts with 'trace.to_' (i.e., this component calling others) and is_anomaly
    from_mask = (trace_win['cmdb_id']==comp) & (trace_win['trace_name'].str.startswith('trace.to_')) & (trace_win['is_anomaly'])
    trace_from_anom = int(from_mask.sum())
    # earliest trace anomaly timestamp where component is involved as either peer target (to) or as source (cmdb_id)
    involved_mask = ((trace_win['peer_service']==comp) | (trace_win['cmdb_id']==comp)) & (trace_win['is_anomaly'])
    if involved_mask.any():
        earliest_trace_ts = trace_win.loc[involved_mask, 'timestamp'].min().tz_convert('UTC').strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        earliest_trace_ts = pd.NA
    rows.append({
        'component': comp,
        'trace_to_anomaly_count': trace_to_anom,
        'trace_from_anomaly_count': trace_from_anom,
        'earliest_trace_anomaly_timestamp_utc': earliest_trace_ts
    })

trace_agg_df = pd.DataFrame(rows)

# Merge metric counts and earliest metric timestamp
summary = pd.DataFrame({'component': candidates})
summary = summary.merge(metric_by_comp.rename(columns={'cmdb_id':'component'}), on='component', how='left')
summary = summary.merge(trace_agg_df, on='component', how='left')
summary = summary.merge(earliest_metric_ts.rename(columns={'cmdb_id':'component'}), on='component', how='left')

# Fill NaNs
summary['metric_anomaly_count'] = summary['metric_anomaly_count'].fillna(0).astype(int)
summary['trace_to_anomaly_count'] = summary['trace_to_anomaly_count'].fillna(0).astype(int)
summary['trace_from_anomaly_count'] = summary['trace_from_anomaly_count'].fillna(0).astype(int)
summary['earliest_metric_anomaly_timestamp_utc'] = summary['earliest_metric_anomaly_timestamp_utc'].fillna(pd.NA)
summary['earliest_trace_anomaly_timestamp_utc'] = summary['earliest_trace_anomaly_timestamp_utc'].fillna(pd.NA)

# is_downstream_leaf = True if trace_to_anomaly_count>0 AND trace_from_anomaly_count==0
summary['is_downstream_leaf'] = (summary['trace_to_anomaly_count']>0) & (summary['trace_from_anomaly_count']==0)

# Order columns as requested
summary = summary.loc[:, [
    'component',
    'metric_anomaly_count',
    'trace_to_anomaly_count',
    'trace_from_anomaly_count',
    'is_downstream_leaf',
    'earliest_metric_anomaly_timestamp_utc',
    'earliest_trace_anomaly_timestamp_utc'
]]

# 3) Ranked list by severity: combine metric_anomaly_count + trace_to_anomaly_count
summary['severity_score'] = summary['metric_anomaly_count'] + summary['trace_to_anomaly_count']
ranked = summary.sort_values('severity_score', ascending=False).reset_index(drop=True)
ranked_list = ranked[['component','severity_score','metric_anomaly_count','trace_to_anomaly_count']]

# Keep outputs compact: one small table (summary) and ranked_list (compact)
summary, ranked_list

```
Out[4]:
```
```python
# Plain-English summary based on the computed aggregates (displayed as a single string)
summary = (
    "Compact summary (incident window 2024-01-20 16:08:45 to 16:38:45 UTC):\n\n"
    "Per-candidate aggregates:\n"
    "- productcatalogservice: metric_anomaly_count=35, trace_to_anomaly_count=147, trace_from_anomaly_count=0, is_downstream_leaf=True, earliest_metric_anomaly=2024-01-20T16:11:00Z, earliest_trace_anomaly=2024-01-20T16:11:00Z\n"
    "- recommendationservice: metric_anomaly_count=60, trace_to_anomaly_count=114, trace_from_anomaly_count=62, is_downstream_leaf=False, earliest_metric_anomaly=2024-01-20T16:11:00Z, earliest_trace_anomaly=2024-01-20T16:11:00Z\n"
    "- currencyservice: metric_anomaly_count=35, trace_to_anomaly_count=114, trace_from_anomaly_count=0, is_downstream_leaf=True, earliest_metric_anomaly=2024-01-20T16:11:00Z, earliest_trace_anomaly=2024-01-20T16:11:00Z\n"
    "- emailservice: metric_anomaly_count=60, trace_to_anomaly_count=85, trace_from_anomaly_count=0, is_downstream_leaf=True, earliest_metric_anomaly=2024-01-20T16:11:00Z, earliest_trace_anomaly=2024-01-20T16:11:00Z\n"
    "- checkoutservice: metric_anomaly_count=23, trace_to_anomaly_count=95, trace_from_anomaly_count=159, is_downstream_leaf=False, earliest_metric_anomaly=2024-01-20T16:11:00Z, earliest_trace_anomaly=2024-01-20T16:11:00Z\n\n"
    "Ranked by combined severity (metric_anomaly_count + trace_to_anomaly_count):\n"
    "1) productcatalogservice (severity_score=182)\n"
    "2) recommendationservice (174)\n"
    "3) currencyservice (149)\n"
    "4) emailservice (145)\n"
    "5) checkoutservice (118)\n\n"
    "Interpretation / guidance:\n"
    "- productcatalogservice is the top candidate by combined severity (high trace_to anomalies) and is a downstream leaf (it is receiving anomalous calls but not calling others), which increases its likelihood as a root-impact node.\n"
    "- currencyservice and emailservice also appear as downstream leaves with high trace_to anomaly counts and should be investigated early.\n"
    "- recommendationservice has high combined severity but is not a downstream leaf (it both receives and issues anomalous traces), so it may be in the middle of the failure propagation.\n"
    "- checkoutservice shows many outgoing anomalous traces (trace_from) and fewer metric anomalies; it appears upstream (calling many services) and may be propagating or exposing issues elsewhere.\n\n"
    "All earliest anomaly timestamps (metrics and traces) were observed at 2024-01-20T16:11:00Z within the window.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               component  metric_anomaly_count  trace_to_anomaly_count  trace_from_anomaly_count  is_downstream_leaf earliest_metric_anomaly_timestamp_utc earliest_trace_anomaly_timestamp_utc  severity_score
0        checkoutservice                    23                      95                       159               False                  2024-01-20T16:11:00Z                 2024-01-20T16:11:00Z             118
1        currencyservice                    35                     114                         0                True                  2024-01-20T16:11:00Z                 2024-01-20T16:11:00Z             149
2           emailservice                    60                      85                         0                True                  2024-01-20T16:11:00Z                 2024-01-20T16:11:00Z             145
3  productcatalogservice                    35                     147                         0                True                  2024-01-20T16:11:00Z                 2024-01-20T16:11:00Z             182
4  recommendationservice                    60                     114                        62               False                  2024-01-20T16:11:00Z                 2024-01-20T16:11:00Z             174,                component  severity_score  metric_anomaly_count  trace_to_anomaly_count
0  productcatalogservice             182                    35                     147
1  recommendationservice             174                    60                     114
2        currencyservice             149                    35                     114
3           emailservice             145                    60                      85
4        checkoutservice             118                    23                      95)```
```